In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt
# Evaluate metrics on the test set
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, matthews_corrcoef

import sys, os
sys.path.insert(0, os.path.join("/home/juni/working/mettl3/notebooks/attention_score/AttentionScore_materials_importable/AttentionScore/", "src"))

# **Load data**

In [ ]:
import numpy as np

import numpy as np

def load_features_labels(path):
    data = np.load(path)
    return data["a"], data["b"]

# train
plec_features_train_all, train_labels = load_features_labels(
    "/home/juni/working/mettl3/notebooks/attention_score/data/plec_features_train_all.npz"
)
ecfp4_features_train_all, train_labels = load_features_labels(
    "/home/juni/working/mettl3/notebooks/attention_score/data/ecfp4_features_train_all.npz"
)
avalon_features_train_all, train_labels = load_features_labels(
    "/home/juni/working/mettl3/notebooks/attention_score/data/avalon_features_train_all.npz"
)

# test
plec_features_test_all, test_labels = load_features_labels(
    "/home/juni/working/mettl3/notebooks/attention_score/data/plec_features_test_all.npz"
)
ecfp4_features_test_all, test_labels = load_features_labels(
    "/home/juni/working/mettl3/notebooks/attention_score/data/ecfp4_features_test_all.npz"
)
avalon_features_test_all, test_labels = load_features_labels(
    "/home/juni/working/mettl3/notebooks/attention_score/data/avalon_features_test_all.npz"
)


# hard test
plec_features_test_all_new, new_test_labels = load_features_labels(
    "/home/juni/working/mettl3/notebooks/attention_score/data/plec_features_test_all_new.npz"
)
ecfp4_features_test_all_new, new_test_labels = load_features_labels(
    "/home/juni/working/mettl3/notebooks/attention_score/data/ecfp4_features_test_all_new.npz"
)
avalon_features_test_all_new, new_test_labels = load_features_labels(
    "/home/juni/working/mettl3/notebooks/attention_score/data/avalon_features_test_all_new.npz"
)

In [ ]:
plec_features_train_all.shape

# **Training with PLEC**

In [ ]:
import numpy as np
from attentionscore.eval.models import default_models
from attentionscore.eval.run import evaluate_models_cv_test

# X_train, y_train, X_test, y_test, (optional) X_hard, y_hard are your arrays
models = default_models(random_state=42, n_jobs=40)

out = evaluate_models_cv_test(
    X_train=plec_features_train_all,
    y_train=train_labels,
    X_test=plec_features_test_all,
    y_test=test_labels,
    X_hard=plec_features_test_all_new,   # optional
    y_hard=new_test_labels,                # optional
    models=models,
    n_splits=5,
    use_test_optimal_threshold=False,      # True = optimistic (peeks at test)
    verbose=True,
)

print("\n==== CV Mean ====")
print(out["cv_mean"].round(3))
print("\n==== CV Std ====")
print(out["cv_std"].round(3))
print("\n==== Test ====")
print(out["test"].round(3))
if out["hard"] is not None:
    print("\n==== Hard Test ====")
    print(out["hard"].round(3))


# **Training with ECFP4**

In [ ]:
import numpy as np
from attentionscore.eval.models import default_models
from attentionscore.eval.run import evaluate_models_cv_test

# X_train, y_train, X_test, y_test, (optional) X_hard, y_hard are your arrays
models = default_models(random_state=42, n_jobs=40)

out = evaluate_models_cv_test(
    X_train=ecfp4_features_train_all,
    y_train=train_labels,
    X_test=ecfp4_features_test_all,
    y_test=test_labels,
    X_hard=ecfp4_features_test_all_new,   # optional
    y_hard=new_test_labels,                # optional
    models=models,
    n_splits=5,
    use_test_optimal_threshold=False,      # True = optimistic (peeks at test)
    verbose=True,
)

print("\n==== CV Mean ====")
print(out["cv_mean"].round(3))
print("\n==== CV Std ====")
print(out["cv_std"].round(3))
print("\n==== Test ====")
print(out["test"].round(3))
if out["hard"] is not None:
    print("\n==== Hard Test ====")
    print(out["hard"].round(3))


# **Training with Avalon**

In [ ]:
import numpy as np
from attentionscore.eval.models import default_models
from attentionscore.eval.run import evaluate_models_cv_test

# X_train, y_train, X_test, y_test, (optional) X_hard, y_hard are your arrays
models = default_models(random_state=42, n_jobs=40)

out = evaluate_models_cv_test(
    X_train=avalon_features_train_all,
    y_train=train_labels,
    X_test=avalon_features_test_all,
    y_test=test_labels,
    X_hard=avalon_features_test_all_new,   # optional
    y_hard=new_test_labels,                # optional
    models=models,
    n_splits=5,
    use_test_optimal_threshold=False,      # True = optimistic (peeks at test)
    verbose=True,
)

print("\n==== CV Mean ====")
print(out["cv_mean"].round(3))
print("\n==== CV Std ====")
print(out["cv_std"].round(3))
print("\n==== Test ====")
print(out["test"].round(3))
if out["hard"] is not None:
    print("\n==== Hard Test ====")
    print(out["hard"].round(3))


In [ ]:
import numpy as np, torch
from attentionscore.nn.train import train_kfold, set_global_seed, evaluate_on_loader
from attentionscore.nn.data import tensors_from_numpy, CustomDataset, make_dataloader

In [ ]:
import numpy as np
import pandas as pd
import os
import math
from torch.utils.data import Dataset, DataLoader,TensorDataset
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.model_selection import KFold

# Evaluate metrics on the test set
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, matthews_corrcoef


In [ ]:
import torch
from torch import nn
class MultiHeadAttention(torch.nn.Module):
    def __init__(self,input_dim,n_heads,ouput_dim=None):
        
        super(MultiHeadAttention, self).__init__()
        self.d_k=self.d_v=input_dim//n_heads
        self.n_heads = n_heads
        if ouput_dim==None:
            self.ouput_dim=input_dim
        else:
            self.ouput_dim=ouput_dim
        self.W_Q = torch.nn.Linear(input_dim, self.d_k * self.n_heads, bias=False)
        self.W_K = torch.nn.Linear(input_dim, self.d_k * self.n_heads, bias=False)
        self.W_V = torch.nn.Linear(input_dim, self.d_v * self.n_heads, bias=False)
        self.fc = torch.nn.Linear(self.n_heads * self.d_v, self.ouput_dim, bias=False)
    def forward(self,X):
        ## (S, D) -proj-> (S, D_new) -split-> (S, H, W) -trans-> (H, S, W)
        Q=self.W_Q(X).view( -1, self.n_heads, self.d_k).transpose(0,1)
        K=self.W_K(X).view( -1, self.n_heads, self.d_k).transpose(0,1)
        V=self.W_V(X).view( -1, self.n_heads, self.d_v).transpose(0,1)
        
        scores = torch.matmul(Q, K.transpose(-1, -2)) / np.sqrt(self.d_k)
        # context: [n_heads, len_q, d_v], attn: [n_heads, len_q, len_k]
        attn = torch.nn.Softmax(dim=-1)(scores)
        context = torch.matmul(attn, V)
        # context: [len_q, n_heads * d_v]
        context = context.transpose(1, 2).reshape(-1, self.n_heads * self.d_v)
        output = self.fc(context)
        return output


# In[107]:


class EncoderLayer(torch.nn.Module):
    def __init__(self,input_dim,n_heads):
        super(EncoderLayer, self).__init__()
        self.attn = MultiHeadAttention(input_dim,n_heads)
        self.AN1=torch.nn.LayerNorm(input_dim)
        
        self.l1=torch.nn.Linear(input_dim, input_dim)
        self.AN2=torch.nn.LayerNorm(input_dim)
    def forward (self,X):
        
        output=self.attn(X)
        X=self.AN1(output+X)
        
        output=self.l1(X)
        X=self.AN2(output+X)
        
        return X


# In[108]:


def gelu(x):
    return x * 0.5 * (1.0 + torch.erf(x / math.sqrt(2.0)))


# In[109]:




# In[112]:

class feature_encoder(torch.nn.Module):  # twin network
    def __init__(self, vector_size,n_heads,n_layers):
        super(feature_encoder, self).__init__()

        self.layers = torch.nn.ModuleList([EncoderLayer(vector_size, n_heads) for _ in range(n_layers)])
        self.AN = torch.nn.LayerNorm(vector_size)

        self.l1 = torch.nn.Linear(vector_size, vector_size // 2)
        self.bn1 = torch.nn.BatchNorm1d(vector_size // 2)

        self.l2 = torch.nn.Linear(vector_size // 2, vector_size // 4)

        self.l3 = torch.nn.Linear(vector_size // 4, vector_size//2)
        self.bn3 = torch.nn.BatchNorm1d(vector_size // 2)

        self.l4 = torch.nn.Linear(vector_size // 2, vector_size )


        self.dr = torch.nn.Dropout(drop_out_rating)

        self.ac = gelu

    def forward(self, X):

        for layer in self.layers:
            X = layer(X)
        X1=self.AN(X)
        X2 = self.dr(self.bn1(self.ac(self.l1(X1))))
        X3 = self.l2(X2)

        X4 = self.dr(self.bn3(self.ac(self.l3(self.ac(X3)))))
        X5 = self.l4(X4)

        return X1,X2,X3,X5
class feature_encoder2(torch.nn.Module):  # twin network
    def __init__(self, vector_size):
        super(feature_encoder2, self).__init__()

        self.l1 = torch.nn.Linear(vector_size, vector_size // 2)
        self.bn1 = torch.nn.BatchNorm1d(vector_size // 2)

        self.l2 = torch.nn.Linear(vector_size // 2, vector_size // 4)
        self.bn2 = torch.nn.BatchNorm1d(vector_size // 4)

        self.dr = torch.nn.Dropout(drop_out_rating)

        self.ac = gelu

    def forward(self, X):

        X = self.dr(self.bn1(self.ac(self.l1(X))))

        X = self.dr(self.bn2(self.ac(self.l2(X))))

        return X
class Model(torch.nn.Module):
    def __init__(self,input_dim_A,input_dim_B, n_heads,n_layers,event_num):
        super(Model, self).__init__()

        #self.input_dim = input_dim
        self.input_dim_A = input_dim_A
        self.input_dim_B = input_dim_B
        self.drugEncoder_input_dim_A=self.input_dim_A
        self.drugEncoder_input_dim_B=self.input_dim_B
        self.drugEncoderA=feature_encoder(self.drugEncoder_input_dim_A,n_heads,n_layers)
        self.drugEncoderB = feature_encoder(self.drugEncoder_input_dim_B, n_heads, n_layers)

        self.feaEncoder1_3_input_dim=self.drugEncoder_input_dim_A+self.drugEncoder_input_dim_B//4
        self.feaEncoder3_1_input_dim=self.drugEncoder_input_dim_B+self.drugEncoder_input_dim_A//4
        self.feaEncoder2_input_dim = self.drugEncoder_input_dim_A//2 + self.drugEncoder_input_dim_B// 2

        self.feaEncoder1 = feature_encoder2(self.feaEncoder1_3_input_dim)
        self.feaEncoder2 = feature_encoder2(self.feaEncoder2_input_dim)
        self.feaEncoder3 = feature_encoder2(self.feaEncoder3_1_input_dim)

        self.feaEncoder1_3_output_dim = self.feaEncoder1_3_input_dim//4
        self.feaEncoder3_1_output_dim = self.feaEncoder3_1_input_dim//4
        self.feaEncoder2_output_dim = self.feaEncoder2_input_dim//4

        self.feaFui_input_dim = self.feaEncoder1_3_output_dim+self.feaEncoder3_1_output_dim+self.feaEncoder2_output_dim+self.drugEncoder_input_dim_A//4+self.drugEncoder_input_dim_B//4

        #self.feaFui = feature_encoder(self.feaFui_input_dim, n_heads, n_layers)
        self.feaFui = feature_encoder2(self.feaFui_input_dim)
        self.linear_input_dim = self.feaFui_input_dim//4+self.feaFui_input_dim

        self.l1=torch.nn.Linear(self.linear_input_dim,(self.linear_input_dim)//2)
        self.bn1=torch.nn.BatchNorm1d((self.linear_input_dim)//2)

        self.l2 = torch.nn.Linear((self.linear_input_dim)//2, 1)
        
        self.ac=gelu

        self.dr = torch.nn.Dropout(drop_out_rating)
        self.sigmoid = torch.nn.Sigmoid()  # Sigmoid for probabilities
        
    def forward(self, XA, XB):
        # XA = X[:, 0:self.input_dim//2]
        # XB = X[:, self.input_dim//2:]

        XA1,XA2,XA3,XAC=self.drugEncoderA(XA)
        XB1, XB2, XB3 ,XBC= self.drugEncoderB(XB)

        XDC = torch.cat((XAC, XBC), 1)

        X1 = torch.cat((XA1,XB3), 1)
        X2 = torch.cat((XA2, XB2), 1)
        X3 = torch.cat((XA3, XB1), 1)

        X1=self.feaEncoder1(X1)
        X2 = self.feaEncoder2(X2)
        X3 = self.feaEncoder3(X3)

        XC = torch.cat((X1, X2, X3,XA3,XB3), 1)
        #_,_,XC,_=self.feaFui(XC)
        XC=self.feaFui(XC)

        X = torch.cat((XA3,XB3,X1, X2, X3,XC), 1)

        X=self.dr(self.bn1(self.ac(self.l1(X))))


        X=self.l2(X)
        X= self.sigmoid(X)
        return X,XC,XDC, XAC, XBC

In [ ]:
class my_loss(nn.Module):
    def __init__(self,classNum):
        
        super(my_loss,self).__init__()
        
        self.criteria1 = torch.nn.BCELoss()
        self.criteria2=torch.nn.MSELoss()

    def forward(self, X, target,XC,XDC,XAC, XBC):
        loss = self.criteria1(X, target) + \
               10*self.criteria2(inputs.float(), XDC)
        return loss

In [ ]:
XA, XB, y = tensors_from_numpy(plec_features_train_all, avalon_features_train_all, train_labels)
XA_test, XB_test, y_test = tensors_from_numpy(plec_features_test_all, avalon_features_test_all, test_labels)
XA_test_new, XB_test_new, y_test_new = tensors_from_numpy(plec_features_test_all_new, avalon_features_test_all_new, new_test_labels)

In [ ]:
# Create the dataset
train_dataset = CustomDataset(XA, XB, y)

# Create DataLoader
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

# Example: Print out the first batch of the data
for batch_idx, (XA_batch, XB_batch, labels) in enumerate(train_loader):
    print(f"Batch {batch_idx + 1} XA shape: {XA_batch.shape}, XB shape: {XB_batch.shape}, Labels shape: {labels.shape}")
    break  # Just print the first batch for demonstration

In [ ]:
# Create the dataset
test_dataset = CustomDataset(XA_test, XB_test, y_test)

# Create DataLoader
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# Example: Print out the first batch of the data
for batch_idx, (XA_batch, XB_batch, labels) in enumerate(test_loader):
    print(f"Batch {batch_idx + 1} XA shape: {XA_batch.shape}, XB shape: {XB_batch.shape}, Labels shape: {labels.shape}")
    break  # Just print the first batch for demonstration


In [ ]:
# Create the dataset
test_dataset_new = CustomDataset(XA_test_new, XB_test_new, y_test_new)

# Create DataLoader
test_loader_new = DataLoader(test_dataset_new, batch_size=128, shuffle=False)

# Example: Print out the first batch of the data
for batch_idx, (XA_batch, XB_batch, labels) in enumerate(test_loader_new):
    print(f"Batch {batch_idx + 1} XA shape: {XA_batch.shape}, XB shape: {XB_batch.shape}, Labels shape: {labels.shape}")
    break  # Just print the first batch for demonstration


In [ ]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")

In [ ]:
drop_out_rating = 0.001
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import KFold
# Example model instantiation
input_dim_A = 4092  # Example dimension for input A
input_dim_B = 512  # Example dimension for input B
n_heads = 1        # Number of attention heads
n_layers = 1       # Number of encoder layers
event_num = 1 
num_epochs = 10

In [ ]:
# ================== Reproducible K-Fold Training (Deterministic) ==================
import os, random
import numpy as np
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Subset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef
)

# -------------------- 1) Determinism block --------------------
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# If you want *strict* determinism, uncomment (may raise if an op isn't deterministic):
# torch.use_deterministic_algorithms(True)

# A dedicated seeded generator for any DataLoader with shuffle=True
g = torch.Generator()
g.manual_seed(SEED)

# -------------------- 2) Build label vector for stratification --------------------
N = len(train_loader.dataset)
y_all = np.array([float(train_loader.dataset[i][2]) for i in range(N)])

# -------------------- 3) Stratified K-Fold (deterministic) --------------------
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
splits = list(kf.split(np.arange(N), y_all))  # save exact splits for reproducibility

# -------------------- 4) Metric helper --------------------
def compute_metrics_from_probs(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true, dtype=float)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= threshold).astype(float)

    acc  = (y_pred == y_true).mean()
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    try:
        roc  = roc_auc_score(y_true, y_prob)          # AUC on probabilities
    except ValueError:
        roc  = float("nan")
    pr   = average_precision_score(y_true, y_prob)    # PR-AUC on probabilities
    mcc  = matthews_corrcoef(y_true, y_pred)
    return acc, prec, rec, f1, roc, pr, mcc

# -------------------- 5) Storage --------------------
fold_train_losses = []
fold_val_losses = []
fold_train_accuracies = []
fold_val_accuracies = []

train_metrics = {k: [] for k in ["accuracy","precision","recall","f1_score","roc_auc","pr_auc","mcc"]}
val_metrics   = {k: [] for k in ["accuracy","precision","recall","f1_score","roc_auc","pr_auc","mcc"]}

# -------------------- 6) K-Fold loop (deterministic) --------------------
for fold, (train_idx, val_idx) in enumerate(splits):
    print(f"\n===== Fold {fold+1}/{len(splits)} =====", flush=True)

    # Reinitialize model and optimizer
    model = Model(input_dim_A, input_dim_B, n_heads, n_layers, event_num).to(device)
    criterion1 = nn.BCELoss()   # keep your loss design (model should output sigmoid probs)
    criterion2 = nn.MSELoss()
    optimizer  = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

    # Create deterministic dataloaders for this fold
    train_subset = Subset(train_loader.dataset, train_idx)
    val_subset   = Subset(train_loader.dataset, val_idx)

    train_loader_fold = DataLoader(
        train_subset, batch_size=64, shuffle=True,
        num_workers=0, generator=g, pin_memory=False
    )
    val_loader_fold = DataLoader(
        val_subset, batch_size=64, shuffle=False,
        num_workers=0, pin_memory=False
    )

    # Epoch-wise logs for this fold
    train_losses = []
    val_losses   = []
    train_accuracies = []
    val_accuracies   = []
    fold_train_metrics = {key: [] for key in train_metrics.keys()}
    fold_val_metrics   = {key: [] for key in val_metrics.keys()}

    for epoch in range(num_epochs):
        # ===== TRAIN (optimize) =====
        model.train()
        running_loss = 0.0
        total_samples = 0

        for XA_batch, XB_batch, targets in train_loader_fold:
            XA_batch = XA_batch.to(device).float()
            XB_batch = XB_batch.to(device).float()
            targets  = targets.to(device).unsqueeze(1)  # (B,1)

            optimizer.zero_grad()
            predictions, XC, XDC, XAC, XBC = model(XA_batch, XB_batch)  # predictions in [0,1]

            loss1 = criterion1(predictions, targets)
            loss2 = criterion2(XA_batch, XAC)
            loss3 = criterion2(XB_batch, XBC)
            total_loss = 1*loss1 + loss2 + loss3

            total_loss.backward()
            optimizer.step()

            running_loss += total_loss.item() * targets.size(0)
            total_samples += targets.size(0)

        # (Optional) training loss as averaged over optimize loop
        train_loss_epoch = running_loss / max(1, total_samples)

        # ===== EVAL on TRAIN (prob-based metrics) =====
        model.eval()
        train_probs, train_trues = [], []
        eval_train_loss = 0.0
        eval_train_count = 0
        with torch.no_grad():
            for XA_batch, XB_batch, targets in train_loader_fold:
                XA_batch = XA_batch.to(device).float()
                XB_batch = XB_batch.to(device).float()
                targets  = targets.to(device).unsqueeze(1)

                predictions, XC, XDC, XAC, XBC = model(XA_batch, XB_batch)

                loss1 = criterion1(predictions, targets)
                loss2 = criterion2(XA_batch, XAC)
                loss3 = criterion2(XB_batch, XBC)
                tot = 1*loss1 + loss2 + loss3
                eval_train_loss += tot.item() * targets.size(0)
                eval_train_count += targets.size(0)

                probs = predictions.detach().squeeze().cpu().numpy()
                lbls  = targets.detach().cpu().numpy().squeeze()
                train_probs.extend(np.atleast_1d(probs))
                train_trues.extend(np.atleast_1d(lbls))

        train_loss_eval_epoch = eval_train_loss / max(1, eval_train_count)
        tr_acc, tr_prec, tr_rec, tr_f1, tr_roc, tr_pr, tr_mcc = compute_metrics_from_probs(train_trues, train_probs)

        # ===== EVAL on VAL =====
        val_probs, val_trues = [], []
        val_loss_sum = 0.0
        val_count = 0
        with torch.no_grad():
            for XA_batch, XB_batch, targets in val_loader_fold:
                XA_batch = XA_batch.to(device).float()
                XB_batch = XB_batch.to(device).float()
                targets  = targets.to(device).unsqueeze(1)

                predictions, XC, XDC, XAC, XBC = model(XA_batch, XB_batch)

                loss1 = criterion1(predictions, targets)
                loss2 = criterion2(XA_batch, XAC)
                loss3 = criterion2(XB_batch, XBC)
                val_loss = 1*loss1 + loss2 + loss3
                val_loss_sum += val_loss.item() * targets.size(0)
                val_count += targets.size(0)

                probs = predictions.detach().squeeze().cpu().numpy()
                lbls  = targets.detach().cpu().numpy().squeeze()
                val_probs.extend(np.atleast_1d(probs))
                val_trues.extend(np.atleast_1d(lbls))

        val_loss_epoch = val_loss_sum / max(1, val_count)
        va_acc, va_prec, va_rec, va_f1, va_roc, va_pr, va_mcc = compute_metrics_from_probs(val_trues, val_probs)

        # ===== Log exactly once per epoch =====
        train_losses.append(train_loss_eval_epoch)   # eval-mode train loss for consistency
        val_losses.append(val_loss_epoch)
        train_accuracies.append(tr_acc)
        val_accuracies.append(va_acc)

        fold_train_metrics["accuracy"].append(tr_acc)
        fold_train_metrics["precision"].append(tr_prec)
        fold_train_metrics["recall"].append(tr_rec)
        fold_train_metrics["f1_score"].append(tr_f1)
        fold_train_metrics["roc_auc"].append(tr_roc)
        fold_train_metrics["pr_auc"].append(tr_pr)
        fold_train_metrics["mcc"].append(tr_mcc)

        fold_val_metrics["accuracy"].append(va_acc)
        fold_val_metrics["precision"].append(va_prec)
        fold_val_metrics["recall"].append(va_rec)
        fold_val_metrics["f1_score"].append(va_f1)
        fold_val_metrics["roc_auc"].append(va_roc)
        fold_val_metrics["pr_auc"].append(va_pr)
        fold_val_metrics["mcc"].append(va_mcc)

        print(
            f"Epoch {epoch+1}/{num_epochs} | "
            f"TrainLoss {train_loss_eval_epoch:.4f} | ValLoss {val_loss_epoch:.4f} | "
            f"Val Acc {va_acc:.4f} | F1 {va_f1:.4f} | ROC-AUC {va_roc:.4f} | PR-AUC {va_pr:.4f}",
            flush=True
        )

    # ===== Store fold-level results =====
    fold_train_losses.append(train_losses)
    fold_val_losses.append(val_losses)
    fold_train_accuracies.append(train_accuracies)
    fold_val_accuracies.append(val_accuracies)

    for key in train_metrics.keys():
        train_metrics[key].append(fold_train_metrics[key])
        val_metrics[key].append(fold_val_metrics[key])

# ================== End reproducible K-Fold ==================


In [ ]:
import pandas as pd
print('Training')
train_metrics_pd = pd.DataFrame(train_metrics)
train_metrics_mean = train_metrics_pd.applymap(lambda x: sum(x) / len(x) if isinstance(x,list) else x)
overallmean = train_metrics_mean.mean()
print(f'PR-AUC: {overallmean[5]:.4f}')
print(f'ROC-AUC: {overallmean[4]:.4f}')
print(f'Precision: {overallmean[1]:.4f}')
print(f'Recall: {overallmean[2]:.4f}')
print(f'F1 Score: {overallmean[3]:.4f}')
print(f'MCC: {overallmean[6]:.4f}')

print('Validation')
val_metrics_pd = pd.DataFrame(val_metrics)
val_metrics_mean = val_metrics_pd.applymap(lambda x: sum(x) / len(x) if isinstance(x,list) else x)
overallmean = val_metrics_mean.mean()
print(f'PR-AUC: {overallmean[5]:.4f}')
print(f'ROC-AUC: {overallmean[4]:.4f}')
print(f'Precision: {overallmean[1]:.4f}')
print(f'Recall: {overallmean[2]:.4f}')
print(f'F1 Score: {overallmean[3]:.4f}')
print(f'MCC: {overallmean[6]:.4f}')

In [ ]:
import pandas as pd
print("test set")# Final model evaluation on a completely different test set
model.eval()
test_running_loss = 0.0
test_correct = 0
test_total = 0
test_predictions = []
test_labels = []
test_output =[]
with torch.no_grad():
    for batch_idx, (XA_batch, XB_batch, targets) in enumerate(test_loader):
        XA_batch = XA_batch.to(device).float()  # Move XA_batch to device and ensure float32
        XB_batch = XB_batch.to(device).float()  # Move XB_batch to device and ensure float32
        targets = targets.to(device) #Move targets to device and reshape
        predictions, XC, XDC , XAC, XBC= model(XA_batch, XB_batch)
        test_output.extend(predictions.cpu().numpy().squeeze().astype(float))
        test_labels.extend(targets.cpu().numpy().squeeze())
        predicted = (predictions.squeeze() >= 0.5).float()
        test_predictions.extend(predicted.cpu().numpy())
        test_correct += (predicted == targets).sum().item()
        test_total += targets.size(0)

# Compute testidation metrics
test_precision = precision_score(test_labels, test_predictions, zero_division=0)
test_recall = recall_score(test_labels, test_predictions, zero_division=0)
test_f1 = f1_score(test_labels, test_predictions, zero_division=0)
test_roc_auc = roc_auc_score(test_labels, test_predictions)
test_pr_auc = average_precision_score(test_labels, test_predictions)
test_mcc = matthews_corrcoef(test_labels, test_predictions)


print(f'PR-AUC: {test_pr_auc:.4f}')
print(f'ROC-AUC: {test_roc_auc:.4f}')
print(f'Precision: {test_precision:.4f}')
print(f'Recall: {test_recall:.4f}')
print(f'F1 Score: {test_f1:.4f}')
print(f'MCC: {test_mcc:.4f}')

test_result = pd.DataFrame({"Predicted_score": test_output})
test_result['Predicted_Activity'] = test_predictions
test_result['Real_Activity'] = test_labels

In [ ]:
import pandas as pd
print("test set")# Final model evaluation on a completely different test set
model.eval()
test_running_loss = 0.0
test_correct = 0
test_total = 0
test_predictions = []
test_labels = []
test_output =[]
with torch.no_grad():
    for batch_idx, (XA_batch, XB_batch, targets) in enumerate(test_loader_new):
        XA_batch = XA_batch.to(device).float()  # Move XA_batch to device and ensure float32
        XB_batch = XB_batch.to(device).float()  # Move XB_batch to device and ensure float32
        targets = targets.to(device) #Move targets to device and reshape
        predictions, XC, XDC , XAC, XBC= model(XA_batch, XB_batch)
        test_output.extend(predictions.cpu().numpy().squeeze().astype(float))
        test_labels.extend(targets.cpu().numpy().squeeze())
        predicted = (predictions.squeeze() >= 0.5).float()
        test_predictions.extend(predicted.cpu().numpy())
        test_correct += (predicted == targets).sum().item()
        test_total += targets.size(0)

# Compute testidation metrics
test_precision = precision_score(test_labels, test_predictions, zero_division=0)
test_recall = recall_score(test_labels, test_predictions, zero_division=0)
test_f1 = f1_score(test_labels, test_predictions, zero_division=0)
test_roc_auc = roc_auc_score(test_labels, test_predictions)
test_pr_auc = average_precision_score(test_labels, test_predictions)
test_mcc = matthews_corrcoef(test_labels, test_predictions)


print(f'PR-AUC: {test_pr_auc:.4f}')
print(f'ROC-AUC: {test_roc_auc:.4f}')
print(f'Precision: {test_precision:.4f}')
print(f'Recall: {test_recall:.4f}')
print(f'F1 Score: {test_f1:.4f}')
print(f'MCC: {test_mcc:.4f}')

new_test_result = pd.DataFrame({"Predicted_score": test_output})
new_test_result['Predicted_Activity'] = test_predictions
new_test_result['Real_Activity'] = test_labels

In [ ]:
import os, torch

save_dir = "path to model destiantion"
os.makedirs(save_dir, exist_ok=True)  # simpler

save_path = os.path.join(save_dir, "model_FullModel_ecfp4.pth")

# (Optional but nice) move to CPU before saving to make the checkpoint portable
model_cpu = {k: v.cpu() for k, v in model.state_dict().items()}

torch.save({
    "epoch": epoch,                                   # current (or best) epoch
    "model_state_dict": model_cpu,                    # state_dict on CPU
    "optimizer_state_dict": optimizer.state_dict(),   # include if you’ll resume training
    "loss": float(running_loss),                      # make sure it’s JSON-serializable
    # "scheduler_state_dict": scheduler.state_dict()  # include if you use one
    # "metadata": {"notes": "..."}                   # anything else you want to remember
}, save_path)

print(f"Model saved to {save_path}")

In [ ]:
import torch
import os

# same path
save_dir = "path to model destiantion"
save_path = os.path.join(save_dir, "model_FullModel_ecfp4.pth")

# 1. Recreate the model architecture
model = Model(input_dim_A, input_dim_B, n_heads, n_layers, event_num).to(device)   # <-- must match the saved model architecture
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)  # or whatever you used

# 2. Load checkpoint
checkpoint = torch.load(save_path, map_location=torch.device("cpu"))  # or "cuda" if GPU

# 3. Restore model weights
model.load_state_dict(checkpoint["model_state_dict"])

# 4. Restore optimizer state (only if resuming training)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

# 5. Restore extra info
epoch = checkpoint["epoch"]
loss = checkpoint["loss"]

print(f"Checkpoint loaded. Epoch={epoch}, Loss={loss:.4f}")
